In [0]:
%pip install openai --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from openai import OpenAI
import pandas as pd
import json

client = OpenAI(
    base_url="https://dbc-820598a7-e451.cloud.databricks.com/serving-endpoints",
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
)

# Load our full analyzed dataset
df = spark.table("default.ghana_gap_analysis").toPandas()
print("Loaded:", len(df), "facilities")

# Build a summary for the LLM to reason over
def build_context():
    total = len(df)
    deserts = df['is_medical_desert'].sum()
    anomalies = df['has_anomaly'].sum()
    
    regional = df.groupby('address_stateOrRegion').agg(
        total=('unique_id','count'),
        deserts=('is_medical_desert','sum')
    ).reset_index()
    regional['pct'] = (regional['deserts']/regional['total']*100).round(1)
    worst = regional.sort_values('pct', ascending=False).head(5)
    
    context = f"""
You are an AI assistant helping NGO planners analyze Ghana healthcare facility data.

DATASET SUMMARY:
- Total facilities: {total}
- Medical deserts (severely under-resourced): {deserts} ({round(deserts/total*100,1)}%)
- Facilities with anomalies/suspicious data: {anomalies}

WORST REGIONS (by % medical desert):
{worst[['address_stateOrRegion','total','deserts','pct']].to_string(index=False)}

ANOMALY BREAKDOWN:
- 80 facilities: No medical data at all
- 55 facilities: Claim surgery but no theater
- 46 facilities: Claim advanced care but no equipment
- 5 facilities: Claim ICU but no ICU equipment
- 3 facilities: Listed as hospital but no capabilities

TOP SUSPICIOUS FACILITIES:
- Saint John of God Hospital, Duayaw Nkwanta: Claims ICU+surgery but no equipment
- Accra Specialist Eye Hospital: Listed as hospital but no capabilities found
- Sam-j Hospital, Accra: Claims surgery+ICU but no supporting equipment

You can answer questions about:
- Which regions need the most help
- Which facilities are suspicious or anomalous  
- Where to deploy doctors or resources
- What capabilities are missing where
"""
    return context

SYSTEM_CONTEXT = build_context()
print("Context built successfully!")
print(SYSTEM_CONTEXT)

Loaded: 1067 facilities
Context built successfully!

You are an AI assistant helping NGO planners analyze Ghana healthcare facility data.

DATASET SUMMARY:
- Total facilities: 1067
- Medical deserts (severely under-resourced): 758 (71.0%)
- Facilities with anomalies/suspicious data: 169

WORST REGIONS (by % medical desert):
address_stateOrRegion  total  deserts   pct
              ASHANTI      6        6 100.0
             Northern      4        4 100.0
                 KEEA      1        1 100.0
                  Oti      1        1 100.0
                   SH      1        1 100.0

ANOMALY BREAKDOWN:
- 80 facilities: No medical data at all
- 55 facilities: Claim surgery but no theater
- 46 facilities: Claim advanced care but no equipment
- 5 facilities: Claim ICU but no ICU equipment
- 3 facilities: Listed as hospital but no capabilities

TOP SUSPICIOUS FACILITIES:
- Saint John of God Hospital, Duayaw Nkwanta: Claims ICU+surgery but no equipment
- Accra Specialist Eye Hospital: Liste

In [0]:
def ask(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Find relevant facilities for the question
    question_lower = question.lower()
    relevant_facilities = []
    
    if any(word in question_lower for word in ['desert', 'underserved', 'worst', 'missing', 'lack']):
        subset = df[df['is_medical_desert'] == True].head(10)
        for _, row in subset.iterrows():
            relevant_facilities.append(f"- {row['name']} ({row['address_city']}, {row['address_stateOrRegion']}) | score: {row['desert_score']}")
    
    if any(word in question_lower for word in ['anomal', 'suspicious', 'fake', 'incorrect', 'wrong']):
        subset = df[df['has_anomaly'] == True].head(10)
        for _, row in subset.iterrows():
            relevant_facilities.append(f"- {row['name']} ({row['address_city']}) | issues: {row['anomalies']}")

    if any(word in question_lower for word in ['icu', 'emergency', 'surgery', 'equipment']):
        subset = df[df['has_equipment'] == True].head(10)
        for _, row in subset.iterrows():
            relevant_facilities.append(f"- {row['name']} ({row['address_city']}, {row['address_stateOrRegion']}) | capabilities: {row['idp_capabilities'][:100]}")

    facility_context = "\n".join(relevant_facilities) if relevant_facilities else "No specific facilities retrieved."

    response = client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=[
            {"role": "system", "content": SYSTEM_CONTEXT},
            {"role": "user", "content": f"""
Relevant facility data:
{facility_context}

Question: {question}

Answer clearly and specifically. Include facility names and regions where relevant.
Suggest concrete actions the NGO can take.
"""}
        ],
        max_tokens=500
    )
    
    answer = response.choices[0].message.content
    print(answer)
    print()
    return answer

# Test with real NGO questions
ask("Which regions in Ghana need the most urgent help?")

Question: Which regions in Ghana need the most urgent help?
--------------------------------------------------
Based on the provided dataset, the regions in Ghana that need the most urgent help are:

1. **Ashanti**: With 100% of its facilities (6 out of 6) being medical deserts, Ashanti region requires immediate attention.
2. **Northern**: Similarly, the Northern region has 100% of its facilities (4 out of 4) being medical deserts, indicating a severe lack of resources.
3. **KEEA**, **Oti**, and **SH** regions also have 100% of their facilities being medical deserts, but with only 1 facility each, the impact might be relatively smaller compared to Ashanti and Northern regions.

Concrete actions the NGO can take:

1. **Deploy doctors and medical staff**: Send medical professionals to these regions, especially Ashanti and Northern, to provide immediate care and support.
2. **Equipment and supply distribution**: Provide essential medical equipment and supplies to facilities in these regio

'Based on the provided dataset, the regions in Ghana that need the most urgent help are:\n\n1. **Ashanti**: With 100% of its facilities (6 out of 6) being medical deserts, Ashanti region requires immediate attention.\n2. **Northern**: Similarly, the Northern region has 100% of its facilities (4 out of 4) being medical deserts, indicating a severe lack of resources.\n3. **KEEA**, **Oti**, and **SH** regions also have 100% of their facilities being medical deserts, but with only 1 facility each, the impact might be relatively smaller compared to Ashanti and Northern regions.\n\nConcrete actions the NGO can take:\n\n1. **Deploy doctors and medical staff**: Send medical professionals to these regions, especially Ashanti and Northern, to provide immediate care and support.\n2. **Equipment and supply distribution**: Provide essential medical equipment and supplies to facilities in these regions to improve their capabilities.\n3. **Facility assessment and upgrading**: Conduct thorough assessm

In [0]:
ask("Which facilities have suspicious or anomalous data that needs investigation?")

Question: Which facilities have suspicious or anomalous data that needs investigation?
--------------------------------------------------
Based on the provided data, the following facilities have suspicious or anomalous data that needs investigation:

1. Accra Specialist Eye Hospital (Accra) - Listed as a hospital but no capabilities found, and no medical data available.
2. Anwiam Hospital (Kumasi) - Claims advanced care but no equipment listed, and claims surgery but no theater mentioned.
3. Asawinso Seventh-day Adventist Clinic (Sefwi Asawinso) - No medical data found, needs field verification.
4. BAMCEC (Bishop Ackon Memorial Christian Eye Centre) (Cape Coast) - Claims surgery but no theater mentioned.
5. Banhart Specialist Hospital (Abesim - Sunyani) - Claims surgery but no theater mentioned.
6. Baptist Medical Centre (Nalerigu) - Claims advanced care but no equipment listed, and claims surgery but no theater mentioned.
7. Battor Catholic Hospital (Battor) - Claims surgery but no t

'Based on the provided data, the following facilities have suspicious or anomalous data that needs investigation:\n\n1. Accra Specialist Eye Hospital (Accra) - Listed as a hospital but no capabilities found, and no medical data available.\n2. Anwiam Hospital (Kumasi) - Claims advanced care but no equipment listed, and claims surgery but no theater mentioned.\n3. Asawinso Seventh-day Adventist Clinic (Sefwi Asawinso) - No medical data found, needs field verification.\n4. BAMCEC (Bishop Ackon Memorial Christian Eye Centre) (Cape Coast) - Claims surgery but no theater mentioned.\n5. Banhart Specialist Hospital (Abesim - Sunyani) - Claims surgery but no theater mentioned.\n6. Baptist Medical Centre (Nalerigu) - Claims advanced care but no equipment listed, and claims surgery but no theater mentioned.\n7. Battor Catholic Hospital (Battor) - Claims surgery but no theater mentioned.\n8. Bodi Anglican Clinic - Ghana (location not specified) - No medical data found, needs field verification.\n9

In [0]:
ask("Where should we deploy an emergency surgery team in Ghana?")

Question: Where should we deploy an emergency surgery team in Ghana?
--------------------------------------------------
To deploy an emergency surgery team effectively in Ghana, we should focus on regions with the highest need and facilities that can support or are currently lacking surgical capabilities. Given the data:

1. **ASHANTI Region**: Despite having 100% of its facilities classified as medical deserts, there are specific hospitals that could benefit from or support an emergency surgery team. The **Adiebeba Specialist Hospital** in Kumasi, Ashanti Region, mentions "Around-the-clock operations" among its capabilities, suggesting it might have some surgical infrastructure in place. Enhancing or supporting this facility could be a strategic move.

2. **Northern Region**: With all its facilities being medical deserts, deploying a team here could significantly impact healthcare delivery. The **Tamale** area, within the Northern Region, hosts the **Aisha Hospital**, which offers "Em

'To deploy an emergency surgery team effectively in Ghana, we should focus on regions with the highest need and facilities that can support or are currently lacking surgical capabilities. Given the data:\n\n1. **ASHANTI Region**: Despite having 100% of its facilities classified as medical deserts, there are specific hospitals that could benefit from or support an emergency surgery team. The **Adiebeba Specialist Hospital** in Kumasi, Ashanti Region, mentions "Around-the-clock operations" among its capabilities, suggesting it might have some surgical infrastructure in place. Enhancing or supporting this facility could be a strategic move.\n\n2. **Northern Region**: With all its facilities being medical deserts, deploying a team here could significantly impact healthcare delivery. The **Tamale** area, within the Northern Region, hosts the **Aisha Hospital**, which offers "Emergency care service" but lacks detailed information on surgical capabilities. Supporting or enhancing surgical cap

In [0]:
# Interactive widget - NGO planners can type any question
question = dbutils.widgets.text("question", "Which regions need the most urgent help?", "Ask a question about Ghana facilities")
dbutils.widgets.get("question")

user_question = dbutils.widgets.get("question")
ask(user_question)

Question: Which regions need the most urgent help?
--------------------------------------------------
The regions that need the most urgent help are ASHANTI, Northern, KEEA, Oti, and SH, as they have the highest percentage of medical deserts, with 100% of facilities in these regions being severely under-resourced.

The NGO can take the following concrete actions:

1. **Deploy doctors and medical equipment to ASHANTI and Northern regions**: These regions have the highest number of medical deserts (6 and 4 facilities, respectively) and should be prioritized for resource allocation.
2. **Conduct a thorough assessment of facilities in KEEA, Oti, and SH regions**: Although these regions have only one medical desert each, it is essential to understand the specific challenges and needs of these facilities to provide targeted support.
3. **Provide training and capacity-building programs for healthcare workers**: In regions with high percentages of medical deserts, it is likely that healthcare 

'The regions that need the most urgent help are ASHANTI, Northern, KEEA, Oti, and SH, as they have the highest percentage of medical deserts, with 100% of facilities in these regions being severely under-resourced.\n\nThe NGO can take the following concrete actions:\n\n1. **Deploy doctors and medical equipment to ASHANTI and Northern regions**: These regions have the highest number of medical deserts (6 and 4 facilities, respectively) and should be prioritized for resource allocation.\n2. **Conduct a thorough assessment of facilities in KEEA, Oti, and SH regions**: Although these regions have only one medical desert each, it is essential to understand the specific challenges and needs of these facilities to provide targeted support.\n3. **Provide training and capacity-building programs for healthcare workers**: In regions with high percentages of medical deserts, it is likely that healthcare workers may lack the necessary skills and knowledge to provide adequate care. The NGO can offer